### Build testgen Python Package

In [54]:
!pip -q install -e ../.

### Imports

In [1]:
import pandas as pd
import json
import os
from tqdm.notebook import tqdm
from testgen.utils import *
from pathlib import Path
from datetime import datetime as dt

### EDA

In [2]:
base_path = Path().cwd().parent

In [10]:
# # read data and rename columns
# df = pd.read_csv(base_path / "data/requirements_full.csv")
# df.columns = ["requirement", "s1", "s2", "s3", "s4", "s5", "s6"]
df = pd.read_excel(base_path / "data/requirements.xlsx")

In [11]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Find Examples

In [13]:
N_EXAMPLES = 1

indexes_to_drop, examples = get_examples_from_df(df, N_EXAMPLES)

print("Data size before examples:", df.shape[0])
print("Number of Examples:", len(indexes_to_drop))
print("Data size after examples:", df.shape[0] - len(indexes_to_drop))

Data size before examples: 195
Number of Examples: 6
Data size after examples: 189


In [15]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e1 in examples.values():
    for e2 in e1:
        examples_txt += f"Requirement: {e2[0]}\n"
        examples_txt += f"Vector: {e2[1]}\n"
        examples_txt += f"Target Sensor/s: {' and '.join(df.columns[1:][[True if x == 1 else False for x in map(int, e2[1][1:-1].split(','))]])}\n"
        examples_txt += "\n"

In [16]:
print("\n".join(examples_txt.split("\n")[-9:]))

Requirement: Power steering systems, whether hydraulic or electronic, must operate effectively in a wide range of environmental conditions, including extreme temperatures
Vector: [0,0,0,0,1]
Target Sensor/s: steering_torque

Requirement: The vehicle control system must dynamically adjust the steering system in response to accelerator pedal inputs during curve negotiation to maintain the desired cornering line
Vector: [1,1,0,0,0]
Target Sensor/s: acceleration_pedal and wheel_steering_angle




In [17]:
# drop indexes used in examples
df.drop(index=indexes_to_drop, inplace=True)

# LLM

In [18]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPrompt

In [19]:
client = openai_client(
    "azure",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    # api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_version='2024-08-01-preview',
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

In [20]:
# t = client.chat.completions.create(
#     model="gpt-4o-mini", messages=[{"role": "user", "content": "Respond: Here"}]
# )
# t.choices[0].message.content

# Run for all Requirements

In [21]:
model_name = "gpt-4o-mini"
temperature = 0.0

results = []
responses = []

for instance in tqdm(df.iterrows()):
    result, response = client_invoke(
        client,
        model_name,
        temperature,
        SystemPrompt,
        Sensors,
        examples_txt,
        UserPrompt,
        instance,
    )

    results.append(result)
    responses.append(response)

0it [00:00, ?it/s]

In [22]:
accuracy = 0
total_tokens = 0
total_completion_tokens = 0
total_time = 0

for r in results:
    accuracy += r["accuracy"]
    total_tokens += r["total_tokens"]
    total_completion_tokens += r["completion_tokens"]
    total_time += r['response_time']

number_of_reqs = len(results)
accuracy /= len(results)
avg_time_per_req = round(total_time / len(results), 6)
avg_token_per_req = total_tokens / len(results)
avg_completion_token_per_req = total_completion_tokens / len(results)

In [23]:
# save results
time = dt.now()

results_path = "results/single_{model}_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=model_name,
    examples=N_EXAMPLES,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "avg_time_per_req": avg_time_per_req,
        "examples": examples,
        "responses": results}, f, indent=4)